# Running MD with native parameters and analyzing the result

The `from_files → nvt → run` path: build a `System`, build a `Recipe` natively
(no hand-authored `.imd` file), run it, and stream energies straight into Python as a
numpy array via `Simulation.run()` — no `.tre` file round-trip.

This notebook uses `EnergyTimeseries` (`gromos.timeseries`) to wrap that array: column
access, block-averaging, dataframes (polars by default), and plots (plotly by
default, matplotlib available too).

In [ ]:
import numpy as np
import gromos.timeseries
from gromos import Topology, Configuration, System, Recipe, Simulation, EnergyTimeseries

REF_DIR = "../../crates/gromos-md/tests/gromosXX_references"

print("dataframe backend:", gromos.timeseries.config.dataframe_backend)
print("plot backend:     ", gromos.timeseries.config.plot_backend)


## 1. Load `aladip_solvated`: alanine dipeptide + SPC shell, SHAKE

This is the system from the end of `01_load_and_inspect.ipynb` — it needs the manual
`Topology.solvate()` composition, since `System.from_files()` can't load an
unsolvated topology against a solvated configuration.

In [ ]:
def load_aladip_solvated():
    topo = Topology(f"{REF_DIR}/shared/aladip.topo")
    topo.solvate(20)
    conf = Configuration(f"{REF_DIR}/shared/aladip.conf")
    return System(topo, conf)

system = load_aladip_solvated()
print(system)


## 2. Two ways to build a `Recipe`

The native factory (`Recipe.nvt(dt, steps, temperature)`) reproduces sane
GROMOS defaults without hand-authoring an `.imd` file. It's compared here against
`Recipe.from_imd()` loading the real, reference-validated `aladip_solvated.in`.

**Constraints, found missing while first building this notebook, since fixed:** the factories
used to have no way to turn SHAKE on, so a factory-built run of a constrained system (like this
one — its solute H-bonds need SHAKE to stay stable at MD timesteps) would silently diverge to
NaN. `Recipe.nve/nvt/npt` take a `constraints="none"|"hbonds"|"allbonds"` argument
(default `"none"`, matching GROMOS's own default and staying backward compatible) that maps to
the `NTC` constraint mode GROMOS itself uses. Demonstrated below.

In [ ]:
factory_recipe = Recipe.nvt(dt=0.002, steps=100, temperature=300.0)
factory_recipe_constrained = Recipe.nvt(
    dt=0.002, steps=100, temperature=300.0, constraints="hbonds"
)
file_recipe = Recipe.from_imd(f"{REF_DIR}/aladip_solvated/aladip_solvated.in")

print("factory (constraints=none, default):", factory_recipe)
print("  .constraints['solute'] =", factory_recipe.constraints["solute"])
print("factory (constraints=hbonds):        ", factory_recipe_constrained)
print("  .constraints['solute'] =", factory_recipe_constrained.constraints["solute"])
print("file (aladip_solvated.in):           ", file_recipe)
print("  .constraints['solute'] =", file_recipe.constraints["solute"])


### 2a. Proof: `constraints="hbonds"` actually fixes the divergence

Run the same system with the factory params from above, `constraints="none"` vs
`"hbonds"`, and watch the temperature. `"none"` reproduces the original bug
(documented, not a regression — GROMOS's own NTC default is 1/`"none"`); `"hbonds"`
is the fix.

In [ ]:
def run_and_sample_temperature(recipe, n_steps):
    sim = Simulation(load_aladip_solvated(), recipe)
    temps = []
    for _ in range(n_steps):
        sim.step(1)
        temps.append(sim.temperature)
    return temps

temps_none = run_and_sample_temperature(factory_recipe, 60)
temps_hbonds = run_and_sample_temperature(factory_recipe_constrained, 60)

print(f'constraints="none":   T[0]={temps_none[0]:.1f} K -> T[-1]={temps_none[-1]:.1f} K'
      f' (diverges — unconstrained solute H-bonds)')
print(f'constraints="hbonds": T[0]={temps_hbonds[0]:.1f} K -> T[-1]={temps_hbonds[-1]:.1f} K'
      f' (stable, tracks the 300 K target)')

assert all(np.isfinite(t) for t in temps_hbonds), "still diverging!"
assert all(t < 500.0 for t in temps_hbonds), "temperature outside sane range"

## 3. Batch run and `EnergyTimeseries`

`sim.run(steps, ene_freq)` runs `steps` MD steps in Rust, sampling every `ene_freq`-th
step, and returns one `(n_frames, 12)` numpy array — no Python-side per-step loop, no
`.tre` file.

In [ ]:
sim = Simulation(system, file_recipe)
energies = sim.run(200, ene_freq=2)
ts = EnergyTimeseries(energies)
print(f"{len(ts)} frames, columns = {gromos.timeseries.COLUMNS}")


## 4. Plot — total / kinetic / potential vs time

In [ ]:
fig = ts.plot("total", "kinetic", "potential")
fig.update_layout(title="aladip_solvated, NVT: total / kinetic / potential energy")
fig


## 5. Plot — energy component breakdown

This is the plot that specifically exercises the fix made while building this
notebook: `bond`/`angle`/`dihedral`/`improper` used to all get lumped into the `bond`
column (a bug in the underlying Rust bonded-force combiner, not just the Python
bindings — `gromos-forces/src/bonded/mod.rs`). All four terms below are real,
independent values now.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
components = ["bond", "angle", "dihedral", "improper", "lj", "coulomb"]
for ax, component in zip(axes.flat, components):
    ax.plot(ts.time, getattr(ts, component))
    ax.set_title(component)
    ax.set_xlabel("time (ps)")
    ax.set_ylabel("kJ/mol")
fig.tight_layout()
fig


In [ ]:
# Sanity check: none of the four bonded terms are silently zero.
for component in ("bond", "angle", "dihedral", "improper"):
    values = getattr(ts, component)
    assert np.any(values != 0.0), f"{component} is all-zero — regression!"
    print(f"{component:10s} mean = {values.mean():8.3f} kJ/mol")


## 6. Block averaging and dataframes

MD samples are correlated in time, so a plain standard error underestimates the true
uncertainty. `block_average()` splits the trajectory into blocks and reports the
standard error of the block means — the error should grow with block size until
autocorrelation is captured, then plateau.

In [ ]:
block_sizes = [1, 2, 4, 8, 16, 32]
means, errors = [], []
for bs in block_sizes:
    mean, err = ts.block_average("total", block_size=bs)
    means.append(mean)
    errors.append(err)
    print(f"block_size={bs:3d}  mean={mean:10.2f}  error={err:8.2f}")


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=block_sizes, y=errors, mode="lines+markers"))
fig.update_layout(
    xaxis_title="block size (frames)",
    yaxis_title="standard error (kJ/mol)",
    xaxis_type="log",
    title="Block-average error vs block size: total energy",
)
fig


In [ ]:
df = ts.to_dataframe()  # polars by default
print(type(df))
df.head()


## 7. Ensemble comparison: NVE vs NVT vs NPT

All three use `Recipe.from_imd()` on already-validated reference `.in`
files (`water_216_box` for NVE, `water_216_nvt` for Berendsen-thermostatted NVT,
`water_216_npt` for Berendsen thermostat + barostat) from the same starting
configuration — stable, reference-tested settings, no factory-constraint gotchas.

None of `temperature`/`volume`/`pressure` are in `sim.run()`'s batch array, so this
section samples them per step with `sim.step(1)` — the same pattern
`test_run_matches_step_loop` uses in the test suite, and a deliberate contrast with
section 3's batched `run()`: batch mode for throughput, per-step mode for
introspection that isn't (yet) in the batch array.

**On `pressure`:** it's only physically meaningful under NPT. `P = (2*KE - virial) /
(3*V)` — the virial term is populated by `PressureCalculation`, which only NPT's
algorithm sequence includes. Under NVE/NVT the getter still returns a number (the
kinetic-only term), but it's not the true mechanical pressure — plotted here anyway,
faded out, specifically to make that gap visible rather than hide it.

In [ ]:
def load_water():
    return System.from_files(
        f"{REF_DIR}/water_216_box/water_216_box.topo",
        f"{REF_DIR}/water_216_box/water_216_box.conf",
    )

ensembles = {
    "NVE": f"{REF_DIR}/water_216_box/water_216_box.in",
    "NVT": f"{REF_DIR}/water_216_nvt/water_216_nvt.in",
    "NPT": f"{REF_DIR}/water_216_npt/water_216_npt.in",
}

n_steps = 150
results = {}
for name, in_file in ensembles.items():
    sim = Simulation(load_water(), Recipe.from_imd(in_file))
    temp, vol, pres = [], [], []
    for _ in range(n_steps):
        sim.step(1)
        temp.append(sim.temperature)
        vol.append(sim.volume)
        pres.append(sim.pressure)
    results[name] = {"temperature": temp, "volume": vol, "pressure": pres}
    print(
        f"{name}: T {temp[0]:6.1f} -> {temp[-1]:6.1f} K   "
        f"V {vol[0]:.4f} -> {vol[-1]:.4f} nm^3   "
        f"P {pres[0]:8.1f} -> {pres[-1]:8.1f} bar"
    )

time = np.arange(1, n_steps + 1) * 0.002


In [ ]:
from plotly.subplots import make_subplots

colors = {"NVE": "#636EFA", "NVT": "#EF553B", "NPT": "#00CC96"}

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("Temperature", "Volume", "Pressure (only meaningful under NPT)"),
    vertical_spacing=0.08,
)
for name, data in results.items():
    fig.add_trace(
        go.Scatter(x=time, y=data["temperature"], mode="lines", name=name,
                    legendgroup=name, line=dict(color=colors[name])),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=time, y=data["volume"], mode="lines", name=name,
                    legendgroup=name, showlegend=False, line=dict(color=colors[name])),
        row=2, col=1,
    )
    # Pressure is only physically meaningful for NPT — fade NVE/NVT to make that visible.
    opacity = 1.0 if name == "NPT" else 0.3
    fig.add_trace(
        go.Scatter(x=time, y=data["pressure"], mode="lines", name=name,
                    legendgroup=name, showlegend=False,
                    line=dict(color=colors[name]), opacity=opacity),
        row=3, col=1,
    )
fig.add_hline(y=300.0, line_dash="dot", row=1, col=1, annotation_text="target T")

fig.update_yaxes(title_text="K", row=1, col=1)
fig.update_yaxes(title_text="nm³", row=2, col=1)
fig.update_yaxes(title_text="bar", row=3, col=1)
fig.update_xaxes(title_text="time (ps)", row=3, col=1)
fig.update_layout(
    height=750,
    title="water_216_box: NVE drifts, NVT thermostats, NPT thermostats + barostats",
)
fig


In [ ]:
# Sanity check: only the barostatted run (NPT) should show real volume fluctuation;
# NVE/NVT have a fixed box (no PressureCalculation/BerendsenBarostat in their sequence).
nve_vol_range = max(results["NVE"]["volume"]) - min(results["NVE"]["volume"])
nvt_vol_range = max(results["NVT"]["volume"]) - min(results["NVT"]["volume"])
npt_vol_range = max(results["NPT"]["volume"]) - min(results["NPT"]["volume"])

assert nve_vol_range == 0.0, "NVE volume should be exactly fixed"
assert nvt_vol_range == 0.0, "NVT volume should be exactly fixed"
assert npt_vol_range > 0.05, f"expected visible NPT volume response, got {npt_vol_range}"
print(f"NPT volume range: {npt_vol_range:.4f} nm^3 (NVE/NVT: fixed, as expected)")


## 8. Save the final structure

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    out_path = os.path.join(tmp, "aladip_solvated_final.cnf")
    system.write(out_path)
    print(f"wrote final structure to {out_path}")
    assert os.path.exists(out_path)


## 9. Energy minimization: potential energy vs time

`Recipe.minimize(steps)` runs real steepest-descent
minimization through `Simulation` — the algorithm sequence swaps in
`SteepestDescent` for the leap-frog integrator and drops the
thermostat/barostat/kinetic-energy calculation entirely (GROMOS convention:
no velocities during EM, `total_energy == potential_energy`).

Uses `aladip_vacuum`'s own starting structure — not an already-minimized
`_em` reference conformation, which would just produce a flat line.

In [ ]:
em_system = System.from_files(
    f"{REF_DIR}/shared/aladip.topo", f"{REF_DIR}/aladip_vacuum/aladip_vacuum.conf"
)
em_recipe = Recipe.minimize(steps=100)
em_sim = Simulation(em_system, em_recipe)
print(em_sim.algorithm_names)

em_energies = em_sim.run(100, ene_freq=1)
em_ts = EnergyTimeseries(em_energies)
print(f"E[0] = {em_ts.potential[0]:.2f} kJ/mol  ->  E[final] = {em_ts.potential[-1]:.2f} kJ/mol")


In [ ]:
fig = em_ts.plot("potential")
fig.update_layout(
    title="aladip_vacuum steepest-descent minimization",
    xaxis_title="time (ps)",
    yaxis_title="potential energy (kJ/mol)",
)
fig


In [ ]:
# Sanity check: real minimization, not the old no-op bug (build_simulation used to
# silently ignore NTEM and run plain leap-frog at dt=0).
assert np.all(np.isfinite(em_ts.potential))
assert em_ts.potential[-1] < em_ts.potential[0], "EM did not decrease potential energy"
# Once dE < DELE (GROMOS convergence tolerance) the algorithm is a documented no-op,
# so the trace plateaus rather than keeps moving — expected, not a bug.
assert np.allclose(em_ts.potential[-5:], em_ts.potential[-1])
print(f"converged: dropped {em_ts.potential[0] - em_ts.potential[-1]:.2f} kJ/mol")


## Summary

- `Recipe.nve/nvt/npt` take `constraints="none"|"hbonds"|"allbonds"`
  (default `"none"`) — fixed in this session; a factory-built run of a constrained
  system used to silently diverge, demonstrated in section 2a.
- `sim.run(steps, ene_freq)` streams energies as a numpy array; `EnergyTimeseries`
  wraps it with plotting (`plotly`/`matplotlib`) and dataframe (`polars`/`pandas`/dict)
  backends, configurable via `gromos.timeseries.config`.
- The bond/angle/dihedral/improper breakdown is real now — fixed in
  `gromos-forces/src/bonded/mod.rs` while building this notebook (the combiner used to
  discard the per-term split before it ever reached Python).
- `sim.temperature` now uses the same constraint-aware degrees-of-freedom count the
  thermostat couples to, not a bare `3*n_atoms` — the two used to silently disagree.
- `Recipe.minimize()` actually minimizes when run through
  `Simulation` (section 9) — it used to silently fall through to leap-frog at `dt=0`,
  a no-op that looked like it ran.